In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
from pedalboard.io import AudioFile
from pedalboard import Pedalboard, Reverb, Chorus

In [23]:
base_path = Path("..")
raw_path = base_path / "data" / "raw"
processed_path = base_path / "data" / "processed"
metadata_path = base_path / "data" / "metadata.csv"
data_path = raw_path / "egfxset" / "Clean"

raw_path, processed_path, metadata_path, data_path

(WindowsPath('../data/raw'),
 WindowsPath('../data/processed'),
 WindowsPath('../data/metadata.csv'),
 WindowsPath('../data/raw/egfxset/Clean'))

#### Диапазоны параметров

In [24]:
PARAM_RANGES = {
    "room_size": (0.0, 1.0),
    "wet_level": (0.0, 0.6),
    "rate_hz": (0.5, 5.0),
    "depth": (0.0, 1.0),
}

PARAM_RANGES

{'room_size': (0.0, 1.0),
 'wet_level': (0.0, 0.6),
 'rate_hz': (0.5, 5.0),
 'depth': (0.0, 1.0)}

#### Случайный выбор параметров

In [25]:
def sample_parameters():
    return {
        "room_size": np.random.uniform(0.0, 1.0),
        "wet_level": np.random.uniform(0.0, 0.6),
        "rate_hz": np.random.uniform(0.5, 5.0),
        "depth": np.random.uniform(0.0, 1.0),
    }

#### Нормализация параметров

In [26]:
def normalize_params(params):
    return {
        "room_size": params["room_size"],
        "wet_level": params["wet_level"] / 0.6,
        "rate_hz": (params["rate_hz"] - 0.5) / 4.5,
        "depth": params["depth"],
    }

def denormalize_params(params):
    return {
            "room_size": params["room_size"],
            "wet_level": params["wet_level"] * 0.6,
            "rate_hz": params["rate_hz"] * 4.5 + 0.5,
            "depth": params["depth"],
        }

#### Обработка аудиосигнала

In [27]:
def apply_fx(audio, sample_rate, params):

    board = Pedalboard([
        Reverb(
            room_size=params["room_size"],
            wet_level=params["wet_level"]
        ),

        Chorus(
            rate_hz=params["rate_hz"],
            depth=params["depth"]
        )
    ])

    wet_audio = board(audio, sample_rate)

    return wet_audio

#### Чтение файлов

In [29]:
audio_files = list(data_path.rglob("*.wav"))
print(len(audio_files))
print(audio_files[:5])

690
[WindowsPath('../data/raw/egfxset/Clean/Bridge/1-0.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-1.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-10.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-11.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-12.wav')]


In [30]:
def get_source_id(audio_path):
    pickup = audio_path.parent.name
    file_id = audio_path.stem

    return f"{pickup}_{file_id}"

variants = 5
test_path = audio_files[0]
source_id = get_source_id(test_path)

for i in range(variants):
    params = sample_parameters()
    norm_params = normalize_params(params)

    print(source_id, i, params, norm_params)

Bridge_1-0 0 {'room_size': 0.5272238122645405, 'wet_level': 0.18240649573435516, 'rate_hz': 3.606777309410318, 'depth': 0.5254388250219104} {'room_size': 0.5272238122645405, 'wet_level': 0.3040108262239253, 'rate_hz': 0.6903949576467373, 'depth': 0.5254388250219104}
Bridge_1-0 1 {'room_size': 0.0642291695301963, 'wet_level': 0.3148213746066517, 'rate_hz': 1.9159669068404162, 'depth': 0.14573557728521225} {'room_size': 0.0642291695301963, 'wet_level': 0.5247022910110862, 'rate_hz': 0.3146593126312036, 'depth': 0.14573557728521225}
Bridge_1-0 2 {'room_size': 0.8430341379989694, 'wet_level': 0.2918404993456756, 'rate_hz': 4.478196492513238, 'depth': 0.5110438779291236} {'room_size': 0.8430341379989694, 'wet_level': 0.4864008322427926, 'rate_hz': 0.8840436650029417, 'depth': 0.5110438779291236}
Bridge_1-0 3 {'room_size': 0.5903706566480624, 'wet_level': 0.09349642533314796, 'rate_hz': 4.031167731482094, 'depth': 0.7823205570724293} {'room_size': 0.5903706566480624, 'wet_level': 0.155827375

In [31]:
test_output_path = processed_path / "test_wet"
test_output_path.mkdir(parents=True, exist_ok=True)